# LEXIS — `doc_scoped` frozen TEST baseline (164-contract held-out split)

**Purpose**: produce the frozen `doc_scoped` baseline on the held-out 164-contract CUAD test split (`evaluation/splits/cuad_split_v1.json`, split=`test`), completely isolated from the 10-contract dev set used throughout development.

**This notebook does NOT tune, rerank, or optimize anything.** Its only job is an uncontaminated measurement. If the 164-contract result differs materially from the 10-contract dev result (`doc_scoped` = Recall@30 0.8813 / MRR 0.5524, `evaluation/reports/cuad_doc_scoped_dev.json`), **that is a finding to report, not something to fix before freezing.**

**Pinned commit**: `f3fe86c34c002a97e1ab827ef9ddcb63c50e0330` on `foundation-remediation`. This notebook checks out that exact SHA (not a moving branch) so "which code produced this number" is never ambiguous — this is the same commit that produced the validated dev-set integration numbers above.

**Isolation design**: two entirely separate Qdrant collections and local bm25 index directories are used — one for a *sanity check* (the 10 dev contracts, freshly re-ingested in this Colab environment) and a different one for the *test run* (the 164 test contracts). Neither touches your local machine's persistent dev collection/bm25 index, and the two Colab-side scopes never touch each other. This also avoids a real correctness issue: `doc_scoped` BM25 scoring uses the corpus's global IDF, so mixing dev and test chunks into one corpus would let the held-out test set's vocabulary statistics influence dev scores (and vice versa).

**Restartability**: ingestion is checkpointed to Google Drive after every contract. If Colab disconnects mid-run, just reconnect and re-run the ingestion cell for that stage — already-ingested contracts are skipped, not redone.

**Sequence**: Stage A (sanity check on dev, must pass) → Stage B (the real test-split run) → Stage C (freeze + download). Do not skip Stage A.

## 0. Setup

In [ ]:
# Mount Drive for persistence: ingest checkpoints and output artifacts survive
# a Colab disconnect here, so a long ingestion run never has to restart from zero.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/lexis_doc_scoped_test_run"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Persistent run directory:", DRIVE_DIR)

In [ ]:
# Pin the exact commit this run measures -- not a branch, so this cell's
# result is reproducible even if foundation-remediation moves later.
PINNED_COMMIT = "f3fe86c34c002a97e1ab827ef9ddcb63c50e0330"

import os
if not os.path.isdir("/content/LEXIS"):
    !git clone https://github.com/Ujjwaljain16/LEXIS.git /content/LEXIS
%cd /content/LEXIS
!git fetch origin
!git checkout {PINNED_COMMIT}
!pip install -q -e .

checked_out = !git rev-parse HEAD
checked_out = checked_out[0].strip()
assert checked_out == PINNED_COMMIT, f"checked out {checked_out}, expected {PINNED_COMMIT}"
print("Checked out and verified pinned commit:", checked_out)

In [ ]:
# Credentials -- prefer Colab's Secrets manager (key icon in the left sidebar)
# over pasting raw values into a cell. Add secrets named QDRANT_URL, QDRANT_API_KEY,
# POSTGRES_URL, then run this cell. Values are never printed or logged.
import os
from google.colab import userdata

os.environ["QDRANT_URL"] = userdata.get("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = userdata.get("QDRANT_API_KEY")
os.environ["POSTGRES_URL"] = userdata.get("POSTGRES_URL")
print("Credentials loaded from Colab Secrets (values not printed).")

In [ ]:
# Environment/package provenance, captured once up front. This is ALSO recorded
# automatically inside every run_eval.py output artifact (evaluation/provenance.py),
# but printing it here lets you sanity-check the environment before spending time.
import subprocess, sys, json

print("Python:", sys.version)
print("Git SHA:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
for pkg in ["sentence-transformers", "bm25s", "qdrant-client", "torch", "numpy"]:
    v = subprocess.run([sys.executable, "-m", "pip", "show", pkg], capture_output=True, text=True).stdout
    line = next((l for l in v.splitlines() if l.startswith("Version:")), "Version: (not found)")
    print(f"{pkg}: {line}")

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print(
    "\nNote on seeds: chunking, embedding (inference), retrieval, and RRF fusion in this "
    "pipeline are all deterministic (no sampling/dropout at inference). The only seed that "
    "applies anywhere in this notebook is eval.seed from config/defaults.yaml, used solely by "
    "the bootstrap-CI analysis in the last cell of Stage B -- not by ingestion or retrieval itself."
)

In [ ]:
# Download the real CUAD v1 dataset (only the QA json).
from huggingface_hub import hf_hub_download

cuad_path = hf_hub_download(
    repo_id="theatticusproject/cuad",
    repo_type="dataset",
    filename="CUAD_v1/CUAD_v1.json",
)
print("CUAD dataset at:", cuad_path)

In [ ]:
# Confirm the frozen split manifest that ships in the pinned commit is the one we
# expect (contract-disjoint, the known dev/test counts) before using it for anything.
import json
from lexis.evaluation.dataset.cuad_loader import CUADLoader
from lexis.evaluation.dataset.cuad_split import contracts_for_split
from lexis.evaluation.splits import assert_disjoint

SPLIT_MANIFEST = "evaluation/splits/cuad_split_v1.json"
manifest = json.load(open(SPLIT_MANIFEST))
print("salt:", manifest["salt"], " fractions:", manifest["fractions"])
print("counts:", json.dumps(manifest["counts"], indent=2))
print("pinned-to-dev contracts (already analysed during development):", len(manifest["pinned"]["dev"]))

raw = CUADLoader().load(cuad_path)
dev_contracts = contracts_for_split(raw, manifest, "dev")
test_contracts = contracts_for_split(raw, manifest, "test")
assert_disjoint({"dev": [c["title"] for c in dev_contracts], "test": [c["title"] for c in test_contracts]})
print(f"Resolved against the real dataset: {len(dev_contracts)} dev / {len(test_contracts)} test contracts, confirmed disjoint.")
assert len(test_contracts) == manifest["counts"]["test"]["contracts"]
assert len(dev_contracts) == manifest["counts"]["dev"]["contracts"]

## Stage A — Sanity check (10-contract dev set, freshly ingested in Colab)

**Do not skip this.** It proves this Colab environment reproduces the already-known dev result (`doc_scoped` Recall@30=0.8813, MRR=0.5524) before spending 30–90+ minutes on the test split. Uses its own isolated Qdrant collection and bm25 directory — never the collection your local machine has been using.

In [ ]:
import os
os.environ["QDRANT_COLLECTION_PRIMARY"] = "chunks_primary_colab_sanity_v1"
os.environ["BM25_INDEX_DIR"] = "/content/LEXIS/data/bm25_index_colab_sanity"
SANITY_CHECKPOINT = f"{DRIVE_DIR}/sanity_ingest_checkpoint.json"
SANITY_OUTPUT = f"{DRIVE_DIR}/cuad_doc_scoped_sanity_colab.json"

!python scripts/setup_collections.py

In [ ]:
# Resumable: if this cell is interrupted, just re-run it -- already-ingested
# contracts are skipped via the checkpoint file on Drive.
!python -m lexis.evaluation.run_eval \
  --benchmark cuad \
  --cuad-path {cuad_path} \
  --num-contracts 10 \
  --max-questions 50 \
  --top-k 30 \
  --protocol doc_scoped \
  --ingest-checkpoint {SANITY_CHECKPOINT} \
  --skip-diagnostics \
  --output {SANITY_OUTPUT}

In [ ]:
# Compare against the known dev-set reference. Prints the exact diff regardless
# of pass/fail -- tiny nonzero diffs here are informative (e.g. CPU vs GPU float
# differences), not something to silently accept or silently fail on.
import json

REFERENCE_RECALL_30 = 0.8813
REFERENCE_MRR = 0.5524
TOLERANCE = 0.0  # strict by default; widen deliberately (and note why) if a real device-difference is found

sanity = json.load(open(SANITY_OUTPUT))
got_recall = round(sanity["mean_recall_at_k"], 4)
got_mrr = round(sanity["mean_reciprocal_rank"], 4)
diff_recall = got_recall - REFERENCE_RECALL_30
diff_mrr = got_mrr - REFERENCE_MRR

print(f"Reference : Recall@30={REFERENCE_RECALL_30}  MRR={REFERENCE_MRR}")
print(f"Colab got : Recall@30={got_recall}  MRR={got_mrr}")
print(f"Diff      : Recall@30={diff_recall:+.4f}  MRR={diff_mrr:+.4f}  (tolerance={TOLERANCE})")
print(f"Cases scored={sanity['num_cases_scored']} excluded={sanity['num_cases_excluded_no_ground_truth']} unmapped={sanity['num_cases_unmapped']}")

sanity_passed = abs(diff_recall) <= TOLERANCE and abs(diff_mrr) <= TOLERANCE
if not sanity_passed:
    raise AssertionError(
        "SANITY CHECK FAILED: this Colab environment did not reproduce the known dev-set "
        "doc_scoped result. STOP -- do not proceed to Stage B until this is understood. "
        "Check: pinned commit correct? credentials pointing at the right services? "
        "unexpected package version drift (see the provenance cell above)?"
    )
print("\nSANITY CHECK PASSED. Safe to proceed to Stage B.")

## Stage B — The frozen TEST baseline (164 held-out contracts)

Only run this if Stage A passed. This is the expensive step (chunking + embedding + ingesting ~164 contracts, then embedding + retrieving for every test-split question) and can take 30–90+ minutes depending on Colab's GPU allocation. **It is safe to re-run the ingestion cell if the session disconnects** — already-ingested contracts are skipped.

In [ ]:
import os
os.environ["QDRANT_COLLECTION_PRIMARY"] = "chunks_primary_colab_test_v1"
os.environ["BM25_INDEX_DIR"] = "/content/LEXIS/data/bm25_index_colab_test"
TEST_CHECKPOINT = f"{DRIVE_DIR}/test_ingest_checkpoint.json"
TEST_OUTPUT_RAW = f"{DRIVE_DIR}/cuad_doc_scoped_test_RAW.json"  # written every run; not yet the frozen artifact

!python scripts/setup_collections.py

In [ ]:
# The test split has 2033 answerable questions total (see the manifest counts
# printed earlier) -- 10000 is just a safe ceiling well above that so every
# mappable test-split question is scored, not an artificial cap.
!python -m lexis.evaluation.run_eval \
  --benchmark cuad \
  --cuad-path {cuad_path} \
  --split-manifest {SPLIT_MANIFEST} \
  --split-name test \
  --max-questions 10000 \
  --top-k 30 \
  --protocol doc_scoped \
  --ingest-checkpoint {TEST_CHECKPOINT} \
  --skip-diagnostics \
  --output {TEST_OUTPUT_RAW}

In [ ]:
# Inspect the raw result BEFORE freezing anything. Report what came out --
# do not adjust the run to chase the dev-set number.
import json

result = json.load(open(TEST_OUTPUT_RAW))
print("Protocol:", result["protocol"])
print("Contracts used:", result["run_config"]["num_contracts"])
print("Cases scored:", result["num_cases_scored"])
print("Cases excluded (no ground truth / no doc scope):", result["num_cases_excluded_no_ground_truth"])
print("Cases unmapped:", result["num_cases_unmapped"])
print("Recall@30:", result["mean_recall_at_k"])
print("MRR:      ", result["mean_reciprocal_rank"])
print("\nFor comparison, the DEV result (not overwritten, kept as its own artifact):")
print("  Recall@30=0.8813  MRR=0.5524")
print("\nGit SHA recorded in this artifact's provenance:", result["provenance"]["git_sha"])
assert result["provenance"]["git_sha"] == PINNED_COMMIT

In [ ]:
# Bootstrap 95% CIs on the raw per-case results (evaluation/stats.py), computed
# here so the frozen artifact ships with them rather than requiring a second pass.
import json
from lexis.evaluation.stats import bootstrap_ci
from lexis.registry.layered_config import load_yaml

eval_cfg = load_yaml("config/defaults.yaml")["eval"]
print("Using eval config: n_resamples=%d alpha=%s seed=%d" % (eval_cfg["bootstrap_resamples"], eval_cfg["alpha"], eval_cfg["seed"]))

result = json.load(open(TEST_OUTPUT_RAW))
recalls = [c["recall_at_k"] for c in result["per_case"]]
rrs = [c["reciprocal_rank"] for c in result["per_case"]]

recall_ci = bootstrap_ci(recalls, n_resamples=eval_cfg["bootstrap_resamples"], alpha=eval_cfg["alpha"], seed=eval_cfg["seed"])
mrr_ci = bootstrap_ci(rrs, n_resamples=eval_cfg["bootstrap_resamples"], alpha=eval_cfg["alpha"], seed=eval_cfg["seed"])

print(f"Recall@30: mean={recall_ci.mean:.4f}  95% CI [{recall_ci.lo:.4f}, {recall_ci.hi:.4f}]  n={recall_ci.n}")
print(f"MRR      : mean={mrr_ci.mean:.4f}  95% CI [{mrr_ci.lo:.4f}, {mrr_ci.hi:.4f}]  n={mrr_ci.n}")

result["bootstrap_ci"] = {
    "recall_at_30": {"mean": recall_ci.mean, "lo": recall_ci.lo, "hi": recall_ci.hi, "n": recall_ci.n,
                      "n_resamples": recall_ci.n_resamples, "alpha": recall_ci.alpha, "seed": recall_ci.seed},
    "mrr": {"mean": mrr_ci.mean, "lo": mrr_ci.lo, "hi": mrr_ci.hi, "n": mrr_ci.n,
            "n_resamples": mrr_ci.n_resamples, "alpha": mrr_ci.alpha, "seed": mrr_ci.seed},
}
json.dump(result, open(TEST_OUTPUT_RAW, "w"), indent=2)
print("\nCIs appended to", TEST_OUTPUT_RAW)

## Stage C — Freeze

Only run this after confirming Stage B completed successfully (no errors, sensible case counts, CIs computed above). This copies the artifact to its canonical path with an explicit `"status": "frozen_test_baseline"` marker, and downloads it. From this point on, `evaluation/reports/cuad_doc_scoped_test.json` is the frozen doc_scoped test baseline: rerank/contextual-prefix/HyPE ablations compare against it, and it is never overwritten by a routine re-run of this notebook without a deliberate decision to re-freeze.

In [ ]:
import json, shutil
from datetime import datetime, timezone

result = json.load(open(TEST_OUTPUT_RAW))
result["status"] = "frozen_test_baseline"
result["frozen_at"] = datetime.now(timezone.utc).isoformat()
result["frozen_from"] = TEST_OUTPUT_RAW

FROZEN_PATH = "evaluation/reports/cuad_doc_scoped_test.json"
os.makedirs(os.path.dirname(FROZEN_PATH), exist_ok=True)
json.dump(result, open(FROZEN_PATH, "w"), indent=2)
shutil.copy(FROZEN_PATH, f"{DRIVE_DIR}/cuad_doc_scoped_test_FROZEN.json")  # Drive copy survives the Colab session ending

print("Frozen:", FROZEN_PATH)
print("Recall@30:", result["mean_recall_at_k"], " 95% CI:", result["bootstrap_ci"]["recall_at_30"])
print("MRR      :", result["mean_reciprocal_rank"], " 95% CI:", result["bootstrap_ci"]["mrr"])

from google.colab import files
files.download(FROZEN_PATH)

### Next steps (do not start automatically)
Download `cuad_doc_scoped_test.json` back into the local repo at `evaluation/reports/cuad_doc_scoped_test.json` (that path is gitignored -- commit it separately/deliberately if you want it in version control, since it's a substantial generated artifact). Report the frozen number to the team for review. Only after that review: begin the rerank / contextual-prefix / HyPE ablation ladder from `LEXIS_FINAL_PLAN.md` section 4, each evaluated against this frozen baseline with `evaluation/run_stats_report.py`.